In [ ]:
"""
Charles Silkin
Feature Selection with Annealing Project Code
FSU Interdisciplinary Data Science Master's Program - Applied Machine Learning Course
14 February 2023
"""

In [ ]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn import metrics

In [ ]:
### NOTE: DATASETS WERE CHANGED TO COMPLETE VARIOUS PARTS.

# Load datasets
y_init = np.loadtxt("Supporting Data Sets/dexter_train.labels", delimiter=',')
x_init = np.loadtxt("Supporting Data Sets/dexter_train.csv")
yt_init = np.loadtxt("Supporting Data Sets/dexter_valid.labels", delimiter=',')
xt_init = np.loadtxt("Supporting Data Sets/dexter_valid.data")

# Print nrows and ncols
print(x_init.shape,y_init.shape,xt_init.shape,yt_init.shape)

In [ ]:
# Standardize the data
sx=np.std(x_init,axis=0)
x_init=x_init[:,sx>0]
xt_init=xt_init[:,sx>0]
sx=np.std(x_init,axis=0)
mx=np.mean(x_init,axis=0)
x_init=(x_init-mx)/sx
xt_init=(xt_init-mx)/sx

# Convert data objects to Torch tensors
x = torch.tensor(x_init).float()
xt = torch.tensor(xt_init).float()
y = torch.tensor(y_init).float()
yt = torch.tensor(yt_init).float()

# Initialize nrows and ncols of training set
n,p=x.shape
# Initialize nrows of test data
nt=xt.shape[0]

# Add column of ones to data
x=torch.cat((torch.ones(n,1),x),dim=1)
xt=torch.cat((torch.ones(nt,1),xt),dim=1)

# Make copies of y (for loss function)
y1 = y.clone()
yt1 = yt.clone()

# Turn values of "-1" into "0"
y[y<0]=0
yt[yt<0]=0

In [ ]:
# Initialize lists for training and test errors, plus k-values
errs=[]
errts=[]
ks=[10,30,100,300,500]

# FSA loss function:
def fsa_loss(y,xb,s,b):
    ## Multiply y*xb
    yxb = y.view(-1)*xb.view(-1)
    ## Insert into logistic loss function
    lxb=torch.log(1+torch.exp(-yxb))
    
    ## Add shrinkage term
    shrinkage = s*torch.sum(b**2)
    
    return torch.mean(lxb)+shrinkage

# FSA gradient function:
def fsa_gradient(x,xb,y,s,b):
    ## Gradient of logistic loss
    n = x.shape[0]
    py = 1/(1+torch.exp(-xb))
    
    grad_log = -x.t()@(y.view(-1)-py.view(-1))/n
    
    ## Derivative of shrinkage term:
    grad_shrink = 2*s*torch.sqrt(torch.sum(b**2))
    
    ## Final gradient:
    return grad_log + grad_shrink

# Variable scheduling function:
def var_sched(k,p,mu,i,n_iter):
    ## Calculate iteration term
    i_term = ((n_iter-2*i)/(2*i*mu+n_iter))
    
    if i_term < 0:
        return int(k)
    else:
        return int(k + ((p-k)*i_term))

# Obtain model for k = 300

## Initialize betas and predictions
betas = torch.zeros(p+1,1)
xb = x@betas

## Initialize parameters
s = 0.001
mu = 100
N_iter = 300
eta = 0.1
k = 300 ### Manually changed to values in k-list

losses = []

# Iterations:
for it in range(N_iter):
    
    ## Compute gradient
    gb = fsa_gradient(x,xb,y,s,betas)
    
    ## Update betas and predicted values
    betas = betas - eta*gb.view(-1,1)
    xb = x@betas
    
    ## Compute loss
    l = fsa_loss(y1,xb,s,betas)
    
    ## Compute variable schedule
    m = var_sched(k,p,mu,it,N_iter)
    
    if m < betas.shape[0]:
        ## Sort betas in descending order
        ord_w = -torch.sort(-torch.abs(betas.view(-1)))[0]
        ## Find threshold
        thr = ord_w[m-1].item()
        ## Get indeces
        j = torch.where(torch.abs(betas.view(-1)) >= thr)[0]
        ## Update x, xt, and betas with only relevant variables
        x = x[:,j]
        xt = xt[:,j]
        betas = betas[j]
    ## Append losses to list
    losses.append(l)

In [ ]:
# Plot Training Loss vs Iteration Number
plt.plot(losses, color = "red")
plt.title("Training Loss vs Iteration Number for k = 300 (DEXTER)")
plt.xlabel("Iteration Number")
plt.ylabel("Training Loss")

# ROC Curve for k = 300 Model

## Initialize "xb" and create ROC curve for training data
xb=x@betas
fpr, tpr, thr = metrics.roc_curve(y,xb)

## Initialize "xtb" and create ROC curve for testing data
xtb=xt@betas
fprt, tprt, thr = metrics.roc_curve(yt,xtb)

## Plot ROC curve
plt.plot(fpr,tpr)
plt.plot(fprt,tprt)
plt.title("ROC Curves -- Model with 300 Features (DEXTER)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.legend(["Training ROC Curve", "Testing ROC Curve"])

# Training and Test Error Computation

## Initialize predictions for training and test data
xb = x@betas
xtb = xt@betas
py=(xb>0).float()
pyt=(xtb>0).float()

## Compute errors
err=1-metrics.accuracy_score(y,py)
errt=1-metrics.accuracy_score(yt,pyt)

## Append errors to lists
errs.append(err)
errts.append(errt)

# Training and Test Error Plot

## Print error lists (to put values in table)
print(errs)
print(errts)

## Plot training and test errors
plt.plot(ks, errs, marker = "o")
plt.plot(ks, errts, marker = "o")
plt.title("Error vs. Number of Features (DEXTER)")
plt.xlabel("Number of Features")
plt.ylabel("Error")
plt.legend(["Training Error", "Testing Error"])